# PRJNA1167724 / GSE278526: extract, check criteria and make a readable summary (R, Google Colab)

Human omental and subcutaneous adipose stromal cells, obese **without** vs **with** type 2 diabetes (T2D). This notebook extracts the archive and cell metadata, checks the four criteria (human adipose, control vs T2D, more than 10,000 cells, endothelial cells) and writes a readable Excel summary.

**Before running**
1. Runtime > Change runtime type > **R**.
2. Mount Google Drive (folder icon on the left). Your shared tar is `Copy of GSE278526_RAW.tar`; the notebook also tries to download it.
3. Run the cells in order. The cell metadata file is downloaded from GEO; if that fails, upload it to Drive and set `META_PATH` in the configuration cell.
4. At the end, download the Excel file and the CSVs from `/content/gse278526_out` (also copied to Drive).

*Status: tested on a synthetic copy of the archive, not on the real files. Paste any error message back to the assistant.*

## 0. Configuration and packages
Edit the paths here if needed. The defaults search Drive, then download.


In [ ]:
TAR_PATH  <- ""     # e.g. "/content/drive/MyDrive/Copy of GSE278526_RAW.tar"; empty = search Drive, then download
META_PATH <- ""     # e.g. "/content/drive/MyDrive/GSE278526_cell_barcodes_metadata.tsv.gz"; empty = download from GEO
GDRIVE_TAR_ID <- "1DCzYrTIU6j_oFXypNCgTUbHyr1f_kM0b"       # your shared Drive copy of the tar
WORK_DIR <- "/content/gse278526_raw"
OUT_DIR  <- "/content/gse278526_out"
MIN_UMI  <- 500            # cells below this total UMI are flagged (not removed)
ENDO_MIN_MARKERS <- 3      # specific endothelial call: >= this many of the 6 core genes detected
ENDO_OVER_OTHER  <- 2      # ... and endothelial score >= this many times the best other lineage score
ENDO_MARKERS <- c("PECAM1", "VWF", "CLDN5", "EGFL7", "FLT1", "PTPRB")
PANELS <- list(
  endothelial           = c(ENDO_MARKERS, "CDH5", "KDR"),
  lymphatic_endothelial = c("PROX1", "LYVE1", "FLT4"),
  adipocyte             = c("ADIPOQ", "PLIN1", "LIPE", "LEP"),
  fibroblast_progenitor = c("PDGFRA", "DCN", "LUM", "COL1A1"),
  immune                = c("PTPRC", "CD163", "MRC1", "CD14"),
  mural                 = c("ACTA2", "RGS5", "MYH11", "PDGFRB"))
# -------------------------------------------------------------------------

for (p in c("Matrix", "data.table", "digest", "writexl")) if (!requireNamespace(p, quietly = TRUE)) install.packages(p, repos = "https://cloud.r-project.org")
suppressPackageStartupMessages({library(Matrix); library(data.table); library(digest); library(writexl)})
dir.create(WORK_DIR, recursive = TRUE, showWarnings = FALSE); dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

## 1. locate / download inputs

In [ ]:
if (TAR_PATH == "") {
  hits <- list.files("/content/drive", pattern = "GSE278526.*RAW\\.tar$", recursive = TRUE, full.names = TRUE)
  if (length(hits)) TAR_PATH <- hits[1]
}
if (TAR_PATH == "" || !file.exists(TAR_PATH)) {
  message("Tar not found in Drive: trying a direct download")
  TAR_PATH <- file.path(WORK_DIR, "GSE278526_RAW.tar")
  ok <- tryCatch({ system(paste("pip install -q gdown && gdown", GDRIVE_TAR_ID, "-O", shQuote(TAR_PATH))); file.exists(TAR_PATH) && file.size(TAR_PATH) > 1e6 }, error = function(e) FALSE)
  if (!ok) { options(timeout = 3600); download.file("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE278nnn/GSE278526/suppl/GSE278526_RAW.tar", TAR_PATH, mode = "wb") }
}
stopifnot(file.exists(TAR_PATH)); cat("Tar:", TAR_PATH, "(", round(file.size(TAR_PATH) / 1e6), "MB )\n")
if (META_PATH == "") {
  hits <- list.files("/content/drive", pattern = "cell_barcodes_metadata", recursive = TRUE, full.names = TRUE)
  if (length(hits)) META_PATH <- hits[1] else {
    META_PATH <- file.path(WORK_DIR, "GSE278526_cell_barcodes_metadata.tsv.gz")
    options(timeout = 1800)
    tryCatch(download.file("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE278nnn/GSE278526/suppl/GSE278526_cell_barcodes_metadata.tsv.gz", META_PATH, mode = "wb"), error = function(e) message("Metadata download failed: ", conditionMessage(e)))
  }
}
have_meta <- file.exists(META_PATH) && file.size(META_PATH) > 100
cat("Cell metadata file:", if (have_meta) META_PATH else "NOT AVAILABLE (continuing without it)", "\n")

## 2. extract archive + file inventory

In [ ]:
untar(TAR_PATH, exdir = WORK_DIR)
files <- list.files(WORK_DIR, recursive = TRUE, full.names = TRUE); files <- files[!file.info(files)$isdir & grepl("GSM", basename(files))]
inv <- data.table(file = basename(files), size_mb = round(file.size(files) / 1e6, 2), sha256 = vapply(files, function(f) digest(f, algo = "sha256", file = TRUE), ""))
cat("Archive files:", nrow(inv), "(expected 12 = 4 libraries x 3 files)\n"); print(inv[, .(file, size_mb)])

## 3. library table (from the GEO sample records; edit only if GEO changes)

In [ ]:
libs <- data.table(gsm = sprintf("GSM%d", 8548235:8548238), library = c("BRI-1456", "BRI-1457", "BRI-1458", "BRI-1459"),
  tissue = c("Omental (visceral) fat", "Omental (visceral) fat", "Subcutaneous abdominal fat", "Subcutaneous abdominal fat"),
  group = c("Obese, no T2D", "Obese with T2D", "Obese, no T2D", "Obese with T2D"),
  geo_label = c("Omental Adipose tissue Stromal Cells - Obese no metabolic disease", "Omental Adipose tissue Stromal Cells - Obese metabolic disease",
                "Subcutaneous abdominal adipose stromal cells - Obese no metabolic disease", "Subcutaneous abdominal adipose stromal cells - Obese metabolic disease"))

## 4. read each library, per-cell QC, lineage scores, endothelial calls

In [ ]:
mito <- "^MT-"; res <- list(); cells <- list()
for (i in seq_len(nrow(libs))) {
  g <- libs$gsm[i]; pick <- function(t) files[grepl(g, files) & grepl(t, files)]
  fb <- pick("barcodes"); ff <- pick("features|genes"); fm <- pick("matrix")
  if (length(fb) != 1 || length(ff) != 1 || length(fm) != 1) { message("Missing files for ", g); next }
  m <- readMM(fm); bc <- fread(fb, header = FALSE)[[1]]; gene <- fread(ff, header = FALSE)[[2]]
  stopifnot(ncol(m) == length(bc), nrow(m) == length(gene)); m <- as(m, "CsparseMatrix")
  umi <- Matrix::colSums(m); ngene <- Matrix::colSums(m > 0)
  mt <- Matrix::colSums(m[grepl(mito, gene), , drop = FALSE]) / pmax(umi, 1)
  present <- ENDO_MARKERS[ENDO_MARKERS %in% gene]
  det <- if (length(present)) Matrix::colSums(m[match(present, gene), , drop = FALSE] > 0) else rep(0, ncol(m))
  keep <- umi >= MIN_UMI
  sc <- sapply(PANELS, function(p) { p <- p[p %in% gene]; if (length(p)) Matrix::colSums(m[match(p, gene), , drop = FALSE]) / pmax(umi, 1) * 1e4 else rep(0, ncol(m)) })
  if (is.null(dim(sc))) sc <- matrix(sc, ncol = length(PANELS)); colnames(sc) <- names(PANELS)
  top <- ifelse(rowSums(sc) == 0, "none", colnames(sc)[max.col(sc, ties.method = "first")])
  best_other <- apply(sc[, colnames(sc) != "endothelial", drop = FALSE], 1, max)
  endo_spec <- keep & det >= ENDO_MIN_MARKERS & top == "endothelial" & sc[, "endothelial"] > 0 & sc[, "endothelial"] >= ENDO_OVER_OTHER * best_other
  sdt <- as.data.table(round(sc, 3)); setnames(sdt, paste0("score_", colnames(sc)))
  cells[[i]] <- cbind(data.table(library = libs$library[i], gsm = g, tissue_geo = libs$tissue[i], group_geo = libs$group[i], barcode = bc,
    bc16 = sub(".*([ACGT]{16}).*", "\\1", bc), total_umi = as.numeric(umi), genes_detected = as.integer(ngene), mito_fraction = round(as.numeric(mt), 5),
    passes_min_umi = keep, core_endothelial_markers_detected = as.integer(det), endothelial_loose = keep & det >= 3, endothelial_specific = endo_spec,
    top_cell_family = top), sdt)
  cat(libs$library[i], ":", length(bc), "cells;", sum(endo_spec), "endothelial (specific)\n"); rm(m); gc(verbose = FALSE)
}
cells <- rbindlist(cells, fill = TRUE)

## 5. cell metadata: print structure, detect columns, join

In [ ]:
meta_cols <- data.table(); donor_col <- tissue_col <- type_col <- NA_character_; matched <- NA_real_
if (have_meta) {
  md <- fread(META_PATH); cat("\nMetadata:", nrow(md), "rows x", ncol(md), "columns\n"); print(names(md)); print(head(md, 3))
  meta_cols <- data.table(column = names(md), example = vapply(md, function(x) as.character(x[1]), ""), n_unique = vapply(md, uniqueN, 1L), n_missing = vapply(md, function(x) sum(is.na(x) | x == ""), 1L))
  is_bc <- vapply(md, function(x) is.character(x) && mean(grepl("[ACGT]{16}", head(x, 2000))) > 0.8, NA)
  bc_col <- if (any(is_bc)) names(md)[which(is_bc)[1]] else NA_character_
  lib_pat <- "BRI-?14[5-9][0-9]|GSM85482[0-9]{2}"
  is_lib <- vapply(md, function(x) is.character(x) && mean(grepl(lib_pat, head(x, 2000))) > 0.5, NA)
  lib_col <- if (any(is_lib)) names(md)[which(is_lib)[1]] else NA_character_
  find <- function(rx, excl = c(bc_col, lib_col)) { h <- grep(rx, names(md), ignore.case = TRUE, value = TRUE); h <- setdiff(h, excl); if (length(h)) h[1] else NA_character_ }
  donor_col <- find("donor|patient|subject|individual|person|demux|hash|genotype"); tissue_col <- find("tissue|depot|site|source|omental|omentum")
  type_col <- find("cell.?type|annot|label|identity|cluster")
  cat("Detected -> barcode:", bc_col, "| library:", lib_col, "| donor:", donor_col, "| tissue:", tissue_col, "| cell type:", type_col, "\n")
  if (!is.na(bc_col)) {
    norm_lib <- function(x) { x <- sub(".*(BRI)-?(14[5-9][0-9]).*", "\\1-\\2", x); x }
    md[, bc16 := sub(".*([ACGT]{16}).*", "\\1", get(bc_col))]
    md[, library := if (!is.na(lib_col)) { v <- get(lib_col); out <- norm_lib(v); gs <- match(sub(".*(GSM85482[0-9]{2}).*", "\\1", v), libs$gsm); ifelse(grepl("^BRI-", out), out, libs$library[gs]) }
                     else if (any(grepl("BRI-?14[5-9][0-9]", get(bc_col)))) norm_lib(get(bc_col)) else NA_character_]
    use_lib <- !all(is.na(md$library)); keycols <- if (use_lib) c("library", "bc16") else "bc16"
    if (!use_lib) message("WARNING: no library column found in metadata; matching on the 16-letter barcode only (ambiguous across libraries)")
    dup <- sum(duplicated(md[, ..keycols])); cat("Duplicate keys in metadata:", dup, "\n")
    mdj <- unique(md, by = keycols); setnames(mdj, setdiff(names(mdj), keycols), paste0("meta_", setdiff(names(mdj), keycols)))
    cells <- merge(cells, mdj, by = keycols, all.x = TRUE, sort = FALSE)
    first_meta <- grep("^meta_", names(cells), value = TRUE)[1]; matched <- round(100 * mean(!is.na(cells[[first_meta]])), 1)
    cat("Cells matched to metadata:", matched, "%\n")
  }
}
tcol <- if (!is.na(tissue_col)) paste0("meta_", tissue_col) else NA_character_
dcol <- if (!is.na(donor_col)) paste0("meta_", donor_col) else NA_character_
ccol <- if (!is.na(type_col)) paste0("meta_", type_col) else NA_character_

## 6. summaries (plain labels)

In [ ]:
by_lib <- cells[, .(cells_measured = .N, cells_with_at_least_min_umi = sum(passes_min_umi), median_umi = median(total_umi), median_genes = median(genes_detected),
  endothelial_loose = sum(endothelial_loose), endothelial_specific = sum(endothelial_specific)), by = .(library, tissue_geo, group_geo)]
by_lib[, pct_endothelial := round(100 * endothelial_specific / cells_measured, 1)]
by_lib <- by_lib[order(library)]
setnames(by_lib, c("Library", "Tissue (GEO record)", "Group (GEO record)", "Cells measured", "Cells passing UMI floor", "Median UMI per cell", "Median genes per cell",
                   "Endothelial cells (loose: >=3 markers)", "Endothelial cells (specific estimate)", "% endothelial (specific)"))
by_group <- cells[, .(cells = .N, endothelial_specific = sum(endothelial_specific)), by = .(group_geo)][, pct := round(100 * endothelial_specific / cells, 1)]
setnames(by_group, c("Group", "Cells measured", "Endothelial cells (specific)", "% endothelial"))
by_donor <- if (!is.na(dcol)) cells[, .(cells = .N, endothelial_specific = sum(endothelial_specific)), by = c("library", "group_geo", dcol)][order(library)] else data.table(note = "No donor column was detected in the cell metadata")
mix <- cells[, .N, by = .(library, top_cell_family)][, pct := round(100 * N / sum(N), 1), by = library]
mix <- dcast(mix, library ~ top_cell_family, value.var = "pct", fill = 0)
tissue_check <- if (!is.na(tcol)) as.data.table(table(GEO_tissue = cells$tissue_geo, Metadata_tissue = cells[[tcol]], useNA = "ifany")) else data.table(note = "No tissue column detected in metadata")
type_tab <- if (!is.na(ccol)) as.data.table(table(Metadata_cell_type = cells[[ccol]], Library = cells$library))[N > 0] else data.table(note = "No cell-type column detected in metadata")

## 7. criteria check

In [ ]:
total_cells <- nrow(cells); total_endo <- sum(cells$endothelial_specific)
donor_n <- if (!is.na(dcol)) uniqueN(cells[[dcol]]) else NA
tissue_mis <- if (!is.na(tcol)) { a <- tolower(cells$tissue_geo); b <- tolower(cells[[tcol]]); sum(!is.na(b) & ((grepl("omen", a) & !grepl("omen", b)) | (grepl("subcut|sq|abdom", a) & grepl("omen", b)))) } else NA
crit <- data.table(
  criterion = c("Human adipose tissue", "Control vs type 2 diabetes", "More than 10,000 cells", "Endothelial cells"),
  verdict = c("Met", if (!is.na(donor_n)) "Met (donor labels found)" else "Partly: group labels are per library, donors not identified",
              if (total_cells > 10000) "Met" else "Not met", if (total_endo > 0) "Present, estimated by markers" else "Not detected"),
  evidence = c("GEO records: omental and subcutaneous abdominal adipose stromal cells, Homo sapiens.",
               paste0("4 libraries: obese without T2D vs obese with T2D (", paste(libs$library, libs$group, sep = "=", collapse = "; "), "). There is no lean control group.",
                      if (!is.na(donor_n)) paste0(" ", donor_n, " distinct values in donor column '", donor_col, "'.") else " Five patients per group were pooled into one library."),
               paste0(format(total_cells, big.mark = ","), " cells measured across 4 libraries (", format(sum(cells$passes_min_umi), big.mark = ","), " with UMI >= ", MIN_UMI, ")."),
               paste0(format(total_endo, big.mark = ","), " endothelial cells by the specific marker rule (", format(sum(cells$endothelial_loose), big.mark = ","), " by the looser rule). Marker-based estimate, not a cell-type annotation.",
                      if (!is.na(ccol)) paste0(" The metadata has a cell-type column ('", type_col, "'); compare it with these estimates in the 'Cell types' sheet.") else "")))
if (!is.na(tissue_mis)) crit <- rbind(crit, data.table(criterion = "Tissue label check", verdict = if (tissue_mis > 0) "MISMATCH between GEO records and metadata" else "Consistent",
  evidence = paste0(format(tissue_mis, big.mark = ","), " cells have a metadata tissue that disagrees with the GEO sample record. See sheet 'Tissue check'.")))
print(crit)

## 8. write outputs

In [ ]:
cells_out <- cells[, which(!vapply(cells, function(x) all(is.na(x)), logical(1))), with = FALSE]
fwrite(crit, file.path(OUT_DIR, "GSE278526_criteria_check.csv")); fwrite(by_lib, file.path(OUT_DIR, "GSE278526_summary_by_library.csv"))
fwrite(cells_out, file.path(OUT_DIR, "GSE278526_cells_clean.csv")); fwrite(meta_cols, file.path(OUT_DIR, "GSE278526_metadata_columns.csv"))
readme <- data.table(`About this file` = c(
  "Public dataset GSE278526 (BioProject PRJNA1167724): human fat tissue stromal cells from people with obesity, with or without type 2 diabetes (T2D).",
  "Cells come from 4 libraries: omental (belly) fat and subcutaneous (under-skin) fat, each for 'obese, no T2D' and 'obese with T2D'. Five people per group were pooled in each library.",
  "Criteria check: whether the dataset meets the four criteria (human adipose, control vs T2D, more than 10,000 cells, endothelial cells).",
  "By library / By group: cells measured and estimated endothelial (blood-vessel lining) cells.",
  "Endothelial cells here are ESTIMATED from marker genes, not identified in the laboratory. 'Specific' needs at least 3 of 6 blood-vessel genes and a blood-vessel signal at least twice any other cell family.",
  "Limits: no lean control; only 4 libraries; individual people can only be separated if the metadata file contains a donor column (see 'By donor').",
  "Tissue labels follow the GEO sample records. If the metadata file disagrees, the 'Tissue check' sheet shows by how much."))
write_xlsx(list(`Read me` = readme, `Criteria check` = crit, `By library` = by_lib, `By group` = by_group, `Cell family mix (%)` = mix, `By donor` = by_donor,
                `Tissue check` = tissue_check, `Cell types` = type_tab, `Metadata columns` = if (nrow(meta_cols)) meta_cols else data.table(note = "metadata not available")),
           file.path(OUT_DIR, "GSE278526_readable_summary.xlsx"))
cat("\nWrote:\n"); print(list.files(OUT_DIR)); cat("Total size (MB):", round(sum(file.size(list.files(OUT_DIR, full.names = TRUE))) / 1e6, 2), "\n")
if (dir.exists("/content/drive/MyDrive")) { d <- "/content/drive/MyDrive/gse278526_out"; dir.create(d, showWarnings = FALSE); file.copy(list.files(OUT_DIR, full.names = TRUE), d, overwrite = TRUE) }